# 01 — Data Cleaning & Merging

Healthcare Imaging Equipment Utilization & Downtime Analysis

**Month 2 milestone (Feb 2023):** load the three raw CSVs, clean them (missing values, date formats, duplicates), and merge into a single analysis-ready table.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'notebooks'))
import pandas as pd
import data_utils as du

equipment_logs, maintenance_tickets, machine_metadata = du.load_raw()
print(equipment_logs.shape, maintenance_tickets.shape, machine_metadata.shape)

(3316, 6) (61, 7) (9, 4)


## Cleaning

- Drop exact duplicate rows
- Parse date columns
- Impute missing `daily_usage_hours` with each machine's own median usage
- Flag maintenance tickets that are still open (no `resolved_date`) and cap their downtime at the observation window end

In [2]:
equipment_logs_clean, eq_stats = du.clean_equipment_logs(equipment_logs)
machine_metadata_clean = du.clean_machine_metadata(machine_metadata)
as_of = equipment_logs_clean['date'].max()
maintenance_tickets_clean, mt_stats = du.clean_maintenance_tickets(maintenance_tickets, as_of)

print('Equipment logs cleaning:', eq_stats)
print('Maintenance tickets cleaning:', mt_stats)

Equipment logs cleaning: {'duplicates_removed': 33, 'missing_usage_imputed': 17}
Maintenance tickets cleaning: {'duplicates_removed': 0, 'still_open_tickets': 3}


## Merge

Join equipment usage logs with machine metadata (age, manufacturer, last service date) to support the age-vs-downtime analysis in notebook 02 and the R regression.

In [3]:
merged = du.merge_all(equipment_logs_clean, maintenance_tickets_clean, machine_metadata_clean)
merged.head()

,machine_id,machine_type,hospital_site,install_date,date,daily_usage_hours,age_years,manufacturer,last_service_date
0,MRI-101,MRI,Riverside General,2016-03-12,2022-04-01,8.11,7.05,GE HealthCare,2023-03-19
1,MRI-101,MRI,Riverside General,2016-03-12,2022-04-02,5.98,7.05,GE HealthCare,2023-03-19
2,MRI-101,MRI,Riverside General,2016-03-12,2022-04-03,3.70,7.05,GE HealthCare,2023-03-19
3,MRI-101,MRI,Riverside General,2016-03-12,2022-04-04,8.84,7.05,GE HealthCare,2023-03-19
4,MRI-101,MRI,Riverside General,2016-03-12,2022-04-05,9.69,7.05,GE HealthCare,2023-03-19


In [4]:
# Persist cleaned/merged tables for the next notebook and the Excel export step
equipment_logs_clean.to_csv('../outputs/exports/equipment_logs_clean.csv', index=False)
maintenance_tickets_clean.to_csv('../outputs/exports/maintenance_tickets_clean.csv', index=False)
machine_metadata_clean.to_csv('../outputs/exports/machine_metadata_clean.csv', index=False)
merged.to_csv('../outputs/exports/merged_equipment_metadata.csv', index=False)
print('Cleaned tables written to outputs/exports/')

Cleaned tables written to outputs/exports/
